## **Step 1**: Install Spark

##**Step 1**: Install Hadoop

###**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

### **Step 3**. Import the lib




In [ ]:
# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDDPractice").getOrCreate()
sc = spark.sparkContext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Step 2**. Use Pyspark SQL Function

We will utilize **.agg()** method - this method lets you pass an aggregate column expression that uses any of the aggregate functions from the pyspark.sql.functions submodule.

The *pyspark.sql.functions* submodule includes a number of useful functions for performing a variety of computations like standard deviations. Note that all the aggregation functions in this submodule take the name of a column in a GroupedData table as demonstrated via the example below

In [ ]:
df = spark.read.option("header", "true").csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Dataset/Sales Info.csv")

In [ ]:
df.show(100)

+-------+---------+-----+
|Company|   Person|Sales|
+-------+---------+-----+
|   ADBE|     Nick|  400|
|   ADBE|  Charles|  240|
|   ADBE|    Frank|  680|
|   SFDC|     Jody| 1200|
|   SFDC|      Lee|  248|
|   SFDC|  Hui-Ann|  286|
|   AMZN|     Dale| 1720|
|   AMZN|Christine|  700|
|    SAP|     Jill|  500|
|    SAP|      Tim|  260|
|    SAP|    Danny| 1500|
|    SAP|    Peter|  700|
|   ORCL|    Brian|  450|
|   ORCL|  Randolp|  290|
|   ORCL|   Florin|  720|
|   ORCL|    James| 1250|
|   ORCL|     Eric| 1400|
|   BABA|   Darren|  350|
|   BABA|   Derick|  475|
|   BABA|   Damien|  585|
|   BABA|    David|  990|
+-------+---------+-----+



In [ ]:
#Display the data types of the columns
df.dtypes

[('Company', 'string'), ('Person', 'string'), ('Sales', 'string')]

## **Step 2.1** Average sales per company


In [ ]:
from pyspark.sql import functions as F

avg_sales_per_company = df.groupBy("Company").agg(F.avg("sales").alias("Avg_Sales"))
avg_sales_per_company.show()

+-------+---------+
|Company|Avg_Sales|
+-------+---------+
|   SFDC|    578.0|
|   ORCL|    822.0|
|    SAP|    740.0|
|   AMZN|   1210.0|
|   ADBE|    440.0|
|   BABA|    600.0|
+-------+---------+



## **Step 2.2** Count of sales per company


In [ ]:
df.groupBy('Company').count().show()

+-------+-----+
|Company|count|
+-------+-----+
|   SFDC|    3|
|   ORCL|    5|
|    SAP|    4|
|   AMZN|    2|
|   ADBE|    3|
|   BABA|    4|
+-------+-----+



##**Step 2.3** Maximum sales amount per company


In [ ]:
from pyspark.sql import functions as F

max_sales_per_company = df.groupBy("Company").agg(F.max("Sales").alias("Max_Sales"))
max_sales_per_company.show()

+-------+---------+
|Company|Max_Sales|
+-------+---------+
|   ADBE|      680|
|   AMZN|      700|
|   BABA|      990|
|   ORCL|      720|
|    SAP|      700|
|   SFDC|      286|
+-------+---------+



##**Step 2.4** Minimum sales amount per company



In [ ]:
# from pyspark.sql import functions as F

# min_sales_per_company = df.groupBy("Company").agg(F.min("Sales").alias("Min_Sales"))
# min_sales_per_company.show()

+-------+---------+
|Company|Min_Sales|
+-------+---------+
|   ADBE|      240|
|   AMZN|     1720|
|   BABA|      350|
|   ORCL|     1250|
|    SAP|     1500|
|   SFDC|     1200|
+-------+---------+



## **Step 2.5** Cumulative sales per company

In [ ]:
# from pyspark.sql.window import Window
# import pyspark.sql.functions as F

# # We'll calculate the cumulative sum of sales per company using a window function
# windowSpec = Window.partitionBy('Company').orderBy('Sales')
# df_with_cumulative_sales = df.withColumn('CumulativeSales', F.sum('Sales').over(windowSpec))
# df_with_cumulative_sales.show()

+-------+---------+-----+---------------+
|Company|   Person|Sales|CumulativeSales|
+-------+---------+-----+---------------+
|   ADBE|  Charles|  240|          240.0|
|   ADBE|     Nick|  400|          640.0|
|   ADBE|    Frank|  680|         1320.0|
|   AMZN|     Dale| 1720|         1720.0|
|   AMZN|Christine|  700|         2420.0|
|   BABA|   Darren|  350|          350.0|
|   BABA|   Derick|  475|          825.0|
|   BABA|   Damien|  585|         1410.0|
|   BABA|    David|  990|         2400.0|
|   ORCL|    James| 1250|         1250.0|
|   ORCL|     Eric| 1400|         2650.0|
|   ORCL|  Randolp|  290|         2940.0|
|   ORCL|    Brian|  450|         3390.0|
|   ORCL|   Florin|  720|         4110.0|
|    SAP|    Danny| 1500|         1500.0|
|    SAP|      Tim|  260|         1760.0|
|    SAP|     Jill|  500|         2260.0|
|    SAP|    Peter|  700|         2960.0|
|   SFDC|     Jody| 1200|         1200.0|
|   SFDC|      Lee|  248|         1448.0|
+-------+---------+-----+---------

## **Step 2.6** Maximum sales amount across all companies




In [ ]:
# # PySpark SQL MAXmax(Sale
# df.createOrReplaceTempView("Company")
# spark.sql("""
#     SELECT MIN(Sales) AS Max_Sales_Amount
#     FROM Company
# """).show()

+----------------+
|Max_Sales_Amount|
+----------------+
|             990|
+----------------+



## **Step 2.7**  Distinct count of sales across all companies

In [ ]:
df.select('Company','Sales').distinct().count()

21

## **Step 2.8**  Standard deviation of the sales across all companies

In [ ]:
std_dev_sales = df.agg(F.stddev('Sales').alias('StandardDeviation')).show()

+------------------+
| StandardDeviation|
+------------------+
|454.57754852065403|
+------------------+



## **Step 2.9** An ordered list of sales amounts from the maximum to the minimum (Optional)

In [ ]:
# ordered_sales = df.select('Sales').orderBy('Sales', ascending=False)
# ordered_sales.show()

+-----+
|Sales|
+-----+
|  990|
|  720|
|  700|
|  700|
|  680|
|  585|
|  500|
|  475|
|  450|
|  400|
|  350|
|  290|
|  286|
|  260|
|  248|
|  240|
| 1720|
| 1500|
| 1400|
| 1250|
+-----+
only showing top 20 rows



## **Step 3**: Using SQL to interface with Spark

### **Step 3.1**: Register the DataFrame as a SQL temporary view so that you can interact with it using SQL commands


In [ ]:
df.createOrReplaceTempView("sales")

### **Step 3.2**: Query the temporary view using SQL

In [ ]:
sqlDF = spark.sql("SELECT * FROM sales")
sqlDF.show()

+-------+---------+-----+
|Company|   Person|Sales|
+-------+---------+-----+
|   ADBE|     Nick|  400|
|   ADBE|  Charles|  240|
|   ADBE|    Frank|  680|
|   SFDC|     Jody| 1200|
|   SFDC|      Lee|  248|
|   SFDC|  Hui-Ann|  286|
|   AMZN|     Dale| 1720|
|   AMZN|Christine|  700|
|    SAP|     Jill|  500|
|    SAP|      Tim|  260|
|    SAP|    Danny| 1500|
|    SAP|    Peter|  700|
|   ORCL|    Brian|  450|
|   ORCL|  Randolp|  290|
|   ORCL|   Florin|  720|
|   ORCL|    James| 1250|
|   ORCL|     Eric| 1400|
|   BABA|   Darren|  350|
|   BABA|   Derick|  475|
|   BABA|   Damien|  585|
+-------+---------+-----+
only showing top 20 rows



In [ ]:
sqlDF1 = spark.sql("SELECT * FROM (SELECT Company, Person, CAST(Sales AS int) as sales FROM sales) ORDER By sales DESC limit 1")
sqlDF1.show()

+-------+------+-----+
|Company|Person|sales|
+-------+------+-----+
|   AMZN|  Dale| 1720|
+-------+------+-----+



### **Step 3.3**: Filter the data so that only records where the sales is greater than 400

In [ ]:
sqlDF1.filter("Sales > 400").show()

+-------+------+-----+
|Company|Person|sales|
+-------+------+-----+
|   AMZN|  Dale| 1720|
+-------+------+-----+



### **Step 3.4**: Filter the dataset to include Sales that have an average Sales of greater than 400 and the company is 'AMZN'

In [ ]:
sqlDF1.filter("Sales > 400").filter("company == 'AMZN'").show()

+-------+------+-----+
|Company|Person|sales|
+-------+------+-----+
|   AMZN|  Dale| 1720|
+-------+------+-----+



# **Step 4**. Aggregating data using GroupBy

### **Step 4.1**: Find the lowest Sales where the company is *Amazon*


In [ ]:
sqlDF1.filter("Company == 'AMZN'").groupBy().min("Sales").show()

+----------+
|min(Sales)|
+----------+
|      1720|
+----------+



### **Step 4.2** Show average sales of that company

In [ ]:
from pyspark.sql.functions import col

# Cast the 'Sales' column to double
sqlDF = sqlDF.withColumn("Sales", col("Sales").cast("double"))

# Now aggregation works
sqlDF.groupBy("Company").avg("Sales").show()

+-------+----------+
|Company|avg(Sales)|
+-------+----------+
|   SFDC|     578.0|
|   ORCL|     822.0|
|    SAP|     740.0|
|   AMZN|    1210.0|
|   ADBE|     440.0|
|   BABA|     600.0|
+-------+----------+



### **Step 4.3** show the distinct count of that company

In [ ]:
# Instead of using df directly, use sqlDF, which still contains the original data.
sqlDF.select('Company','Sales').distinct().show()

+-------+------+
|Company| Sales|
+-------+------+
|   AMZN|1720.0|
|   AMZN| 700.0|
|    SAP| 260.0|
|   SFDC|1200.0|
|   ORCL| 450.0|
|   BABA| 475.0|
|   ADBE| 680.0|
|    SAP| 500.0|
|   SFDC| 286.0|
|   ORCL| 720.0|
|   BABA| 585.0|
|   ADBE| 240.0|
|    SAP|1500.0|
|    SAP| 700.0|
|   ORCL|1400.0|
|   ORCL|1250.0|
|   BABA| 350.0|
|   BABA| 990.0|
|   SFDC| 248.0|
|   ORCL| 290.0|
+-------+------+
only showing top 20 rows



### **Step 4.4** Compute the standard deviation of ratings count by publisher

In [ ]:
# Import pyspark.sql.functions as F
import pyspark.sql.functions as F

# Standard deviation of ratings count by companies
df = sqlDF.groupBy("Company").agg(F.stddev("Sales"))
df.show()

+-------+------------------+
|Company|     stddev(Sales)|
+-------+------------------+
|   SFDC| 539.0027829241701|
|   ORCL|  487.103685060994|
|    SAP| 537.6492040974921|
|   AMZN| 721.2489168102785|
|   ADBE|222.71057451320087|
|   BABA|  277.158197906298|
+-------+------------------+



# **Step 5**. Joining datasets using Spark SQL

In [ ]:
# Create the tables that will be joined

valuesA = [('Dog',1),('Monkey',2),('Elephant',3),('Penguin',4)]
TableA = spark.createDataFrame(valuesA,['name','id'])

valuesB = [('Lizard',1),('Dog',2),('Elephant',3),('Rat',4)]
TableB = spark.createDataFrame(valuesB,['name','id'])

TableA.show()
TableB.show()

+--------+---+
|    name| id|
+--------+---+
|     Dog|  1|
|  Monkey|  2|
|Elephant|  3|
| Penguin|  4|
+--------+---+

+--------+---+
|    name| id|
+--------+---+
|  Lizard|  1|
|     Dog|  2|
|Elephant|  3|
|     Rat|  4|
+--------+---+



In [ ]:
# Create table aliases
T1 = TableA.alias('T1')
T2 = TableB.alias('T2')

## **Step 5.1** Inner Join

In [ ]:
# Inner Join on name - returns matching records from the 2 tables respectively
T1.join(T2, T1.name == T2.name, 'inner').show()

+--------+---+--------+---+
|    name| id|    name| id|
+--------+---+--------+---+
|     Dog|  1|     Dog|  2|
|Elephant|  3|Elephant|  3|
+--------+---+--------+---+



## **Step5.2** Left Join

In [ ]:
# Left Join Example - If you want to select all records from table 1 and return data from table 2 when it matches, you choose ‘left’
T1.join(T2, T1.name == T2.name, 'left').show()

+--------+---+--------+----+
|    name| id|    name|  id|
+--------+---+--------+----+
|  Monkey|  2|    NULL|NULL|
|     Dog|  1|     Dog|   2|
|Elephant|  3|Elephant|   3|
| Penguin|  4|    NULL|NULL|
+--------+---+--------+----+



In [ ]:
# Left Join - filtering out nulls
left_join = T1.join(T2, T1.name == T2.name, 'left')
left_join.filter((T2.name).isNotNull()).show()

+--------+---+--------+---+
|    name| id|    name| id|
+--------+---+--------+---+
|     Dog|  1|     Dog|  2|
|Elephant|  3|Elephant|  3|
+--------+---+--------+---+



## **Step 5.3** Right Join

In [ ]:
# Right Join Example - If you want to select all records from table 2 and return data from table 1 when it matches, you choose ‘right’
T1.join(T2, T1.name == T2.name, 'right').show()

+--------+----+--------+---+
|    name|  id|    name| id|
+--------+----+--------+---+
|    NULL|NULL|  Lizard|  1|
|     Dog|   1|     Dog|  2|
|Elephant|   3|Elephant|  3|
|    NULL|NULL|     Rat|  4|
+--------+----+--------+---+



In [ ]:
# Right Join - filtering out nulls
Right_join = T1.join(T2, T1.name == T2.name, 'right')
Right_join.filter((T2.name).isNotNull()).show()

+--------+----+--------+---+
|    name|  id|    name| id|
+--------+----+--------+---+
|    NULL|NULL|  Lizard|  1|
|     Dog|   1|     Dog|  2|
|Elephant|   3|Elephant|  3|
|    NULL|NULL|     Rat|  4|
+--------+----+--------+---+



## **Step 5.4** Full Join

In [ ]:
# Full Outer Join - This shows all records from the left table and all the records from the right table and nulls where the two do not match
T1.join(T2, T1.name == T2.name,how='full').show()

+--------+----+--------+----+
|    name|  id|    name|  id|
+--------+----+--------+----+
|     Dog|   1|     Dog|   2|
|Elephant|   3|Elephant|   3|
|    NULL|NULL|  Lizard|   1|
|  Monkey|   2|    NULL|NULL|
| Penguin|   4|    NULL|NULL|
|    NULL|NULL|     Rat|   4|
+--------+----+--------+----+



## Student Practice

Complete the Spark SQL practice questions using the existing Google Drive / Google Cloud setup.

In [ ]:
# Practice 1: Create or replace a temporary SQL view from the main DataFrame
# TODO: df.createOrReplaceTempView("your_table_name")

In [ ]:
# Practice 2: Write a Spark SQL query with SELECT, WHERE, and ORDER BY
# TODO: Use spark.sql(""" ... """)

In [ ]:
# Practice 3: Write a GROUP BY query with COUNT or AVG
# TODO: Summarize one important business question

In [ ]:
# Practice 4: Convert the SQL result back to a DataFrame and display it
# TODO: result_df.show()